In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchinfo import summary

import numpy as np

import pandas as pd

from matplotlib import pyplot as plt

from utils.utils import count_parameters, train, test
from utils.nn_arch import SimpleCNN, MidCNN, Medium_MLP

import random
import os

In [ ]:
def set_seed(seed=212):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Force deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optional: ensure all operations are deterministic
    # torch.use_deterministic_algorithms(True)
    
    # Set hash seed for python
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(28)

In [ ]:
# Download training data from open datasets.
training_data = datasets.MNIST(
    root="data",
    train=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.MNIST(
    root="data",
    train=False,
    transform=ToTensor(),
)

In [ ]:
from torch.utils.data import random_split

# Load the full training set
dataset_full = datasets.MNIST(root='data', train=False, transform=ToTensor())

# Define split lengths
train_size = 500
val_size = 9500
training_data, test_data = random_split(dataset_full, [train_size, val_size])


In [ ]:
batch_size = 16
epochs = 40
lr = 5e-4

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

model = SimpleCNN().to(device)
#model = MidCNN().to(device)
#model = Medium_MLP().to(device)
print(model)

In [ ]:
print('Model total params and trainable params: ', count_parameters(model))
X, y = next(iter(train_dataloader))
summary(model, input_size=(batch_size, X.shape[1], X.shape[2], X.shape[3]))

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

In [ ]:
loss_train = []
loss_test = []
acc_test = []
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    loss_tr = train(train_dataloader, model, loss_fn, optimizer, device=device, return_loss=True)
    loss_train.append(loss_tr)
    loss_te, acc = test(test_dataloader, model, loss_fn, device=device)
    loss_test.append(loss_te)
    acc_test.append(acc)
print("Done!")

In [ ]:
plt.plot(range(1,epochs+1), loss_train, label='Train Loss')
plt.plot(range(1,epochs+1), loss_test, label='Val Loss')
plt.title('Loss vs. Epoch')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.ylim(0,2.5)
plt.legend()

In [ ]:
plt.plot(range(1,epochs+1), acc_test)
plt.title('Accuracy on Validation Set vs. Epoch')
plt.xlabel('Epoch')
plt.ylabel('Classification Accuracy')
plt.ylim(0,1)

In [ ]:
data_train = pd.DataFrame()

data_train['epoch'] = range(1,epochs+1)
data_train['train_loss'] = loss_train
data_train['test_loss'] = loss_test
data_train['test_acc'] = acc_test

print(data_train.head())

data_train.to_csv('cnn/fscnn_lr5e-4_epoch40_base.csv')

In [ ]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

In [ ]:
model = SimpleCNN().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))